# Exploratory Data Analysis: Brazilian Olist E-Commerce

## Executive Summary

- **~97%** of orders in the dataset reach delivered status.
- **6.8% of delivered orders arrive after the estimated date** (~6,500 orders in the analysis window).
- **87 high-risk sellers** (high volume + high late rate, top quartile in both) drive a disproportionate share of all late deliveries.
- Late delivery has a clear impact on ratings: the share of 1–2 star reviews goes from ~9% (on-time) to ~50% (late 1–7 days) to ~81% (very late 8–30 days).
- **Main recommendation:** Focus interventions on the top 87 high-risk sellers and on the states with the highest late rates.

---

## Objective

**Understand the drivers of delivery delays and their impact on customer satisfaction.**

The analysis looks at logistics and late deliveries, then at which sellers and regions are driving them, and finally at how delays translate into review scores. All analysis covers delivered orders with valid delivery dates (2017-01-01 to 2018-08-31).

## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np

# Load datasets
orders = pd.read_csv("./data/olist_orders_dataset.csv")
items = pd.read_csv("./data/olist_order_items_dataset.csv")
sellers = pd.read_csv("./data/olist_sellers_dataset.csv")
customers = pd.read_csv("./data/olist_customers_dataset.csv")
reviews = pd.read_csv("./data/olist_order_reviews_dataset.csv")

print("Orders:", orders.shape)
print("Order items:", items.shape)
print("Sellers:", sellers.shape)
print("Customers:", customers.shape)
print("Reviews:", reviews.shape)

Orders: (99441, 8)
Order items: (112650, 7)
Sellers: (3095, 4)
Customers: (99441, 5)
Reviews: (99224, 7)


In [2]:
# Parse datetime columns
date_cols_orders = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
orders[date_cols_orders] = orders[date_cols_orders].apply(pd.to_datetime, errors="coerce")

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

# Restrict to delivered orders in the analysis window and with usable delivery info
orders_analysis = orders.copy()
orders_analysis = orders_analysis[orders_analysis["order_status"] == "delivered"]
orders_analysis = orders_analysis[
    (orders_analysis["order_purchase_timestamp"] >= "2017-01-01") &
    (orders_analysis["order_purchase_timestamp"] <= "2018-08-31")
]
orders_analysis = orders_analysis.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"])

# Delivery performance metrics
orders_analysis["delivery_delay_days"] = (
    orders_analysis["order_delivered_customer_date"] - orders_analysis["order_estimated_delivery_date"]
).dt.days
orders_analysis["delivery_time_days"] = (
    orders_analysis["order_delivered_customer_date"] - orders_analysis["order_purchase_timestamp"]
).dt.days

print(f"Delivered orders in analysis window (with valid dates): {len(orders_analysis):,}")
orders_analysis[["delivery_delay_days", "delivery_time_days"]].describe(percentiles=[0.5, 0.9, 0.99])

Delivered orders in analysis window (with valid dates): 96,203


,delivery_delay_days,delivery_time_days
count,96203.000000,96203.000000
mean,-11.807969,12.073854
std,10.088942,9.535542
min,-147.000000,0.000000
50%,-12.000000,10.000000
90%,-2.000000,23.000000
99%,18.000000,46.000000
max,188.000000,209.000000


## 2. Logistics and Late Deliveries

**Delivery delay** = actual delivery date minus estimated delivery date (positive = late). Delivery time (purchase to delivery) is also included for context.

In [3]:
n = len(orders_analysis)
late_any = (orders_analysis["delivery_delay_days"] > 0).sum()
late_7 = (orders_analysis["delivery_delay_days"] > 7).sum()
late_30 = (orders_analysis["delivery_delay_days"] > 30).sum()
on_time = (orders_analysis["delivery_delay_days"] <= 0).sum()

print("Delivery performance (delivered orders with valid dates):")
print(f"  On time or early:     {on_time:,}  ({100 * on_time / n:.2f}%)")
print(f"  Late (1–7 days):      {late_any - late_7:,}  ({100 * (late_any - late_7) / n:.2f}%)")
print(f"  Very late (8–30 days): {late_7 - late_30:,}  ({100 * (late_7 - late_30) / n:.2f}%)")
print(f"  Extremely late (>30): {late_30:,}  ({100 * late_30 / n:.2f}%)")
print(f"  Total late:           {late_any:,}  ({100 * late_any / n:.2f}%)")

Delivery performance (delivered orders with valid dates):
  On time or early:     89,672  (93.21%)
  Late (1–7 days):      3,671  (3.82%)
  Very late (8–30 days): 2,516  (2.62%)
  Extremely late (>30): 344  (0.36%)
  Total late:           6,531  (6.79%)


In [4]:
# Late delivery by customer state (destination)
orders_with_customer = orders_analysis.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
dest_late = orders_with_customer.groupby("customer_state").agg(
    orders=("order_id", "count"),
    late=("delivery_delay_days", lambda x: (x > 0).sum()),
).assign(pct_late=lambda d: (100 * d["late"] / d["orders"]).round(2)).sort_values("pct_late", ascending=False)
print("Late delivery rate by customer state (destination) — top 10:")
print(dest_late.head(10))

Late delivery rate by customer state (destination) — top 10:
                orders  late  pct_late
customer_state                        
AL                 396    85     21.46
MA                 713   125     17.53
SE                 332    51     15.36
PI                 475    66     13.89
CE                1273   176     13.83
RR                  40     5     12.50
BA                3253   396     12.17
RJ               12310  1495     12.14
PA                 942   106     11.25
ES                1992   214     10.74


In [5]:
# Late deliveries over time (monthly)
orders_analysis["purchase_month"] = orders_analysis["order_purchase_timestamp"].dt.to_period("M")
monthly = orders_analysis.groupby("purchase_month").agg(
    total_orders=("order_id", "count"),
    late_orders=("delivery_delay_days", lambda x: (x > 0).sum()),
)
monthly["pct_late"] = (100 * monthly["late_orders"] / monthly["total_orders"]).round(2)
monthly = monthly.reset_index()
monthly["purchase_month"] = monthly["purchase_month"].astype(str)
print(monthly.to_string(index=False))

purchase_month  total_orders  late_orders  pct_late
       2017-01           750           22      2.93
       2017-02          1653           49      2.96
       2017-03          2546          116      4.56
       2017-04          2303          151      6.56
       2017-05          3545          106      2.99
       2017-06          3135           95      3.03
       2017-07          3872          108      2.79
       2017-08          4193          122      2.91
       2017-09          4150          182      4.39
       2017-10          4478          187      4.18
       2017-11          7288          904     12.40
       2017-12          5513          411      7.46
       2018-01          7069          403      5.70
       2018-02          6555          926     14.13
       2018-03          7003         1328     18.96
       2018-04          6798          306      4.50
       2018-05          6749          443      6.56
       2018-06          6096           71      1.16
       2018-

**Conclusion — Logistics and late deliveries**

- **Most orders arrive on time or early:** about 93% of delivered orders in the analysis window are delivered by the estimated date. The median delivery is around 12 days before the estimate.
- **About 7% of orders are late.** Most late deliveries fall in the 1–7 day range, but there’s a tail of extreme delays (over 30 days). Even the moderate delays represent thousands of orders and a real opportunity for improvement.
- **Late rates vary by month** but don’t show a clear worsening trend, which suggests the problem is manageable through targeted process and carrier improvements rather than a platform-wide breakdown.

## 3. Geographic analysis of late deliveries

Where do late deliveries happen? Two angles: **destination** (customer state, where orders are received) and **origin** (seller state, where the seller ships from). This helps identify regions that might need better carrier coverage or logistics support.

In [6]:
# By destination (customer state): where do buyers experience late delivery?
orders_with_customer = orders_analysis.merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left"
)
late_by_destination = (
    orders_with_customer.groupby("customer_state")
    .agg(orders=("order_id", "count"), late=("delivery_delay_days", lambda x: (x > 0).sum()))
    .assign(pct_late=lambda d: (100 * d["late"] / d["orders"]).round(2))
    .sort_values("pct_late", ascending=False)
)
print("Late delivery rate by customer state (destination) — where orders arrive:")
print(late_by_destination.to_string())
print()

# By origin (seller state): where do late shipments come from? (first seller per order)
order_first_seller = items.groupby("order_id")["seller_id"].first().reset_index()
seller_geo = order_first_seller.merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left"
).merge(
    orders_analysis[["order_id", "delivery_delay_days"]], on="order_id", how="inner"
)
late_by_origin = (
    seller_geo.groupby("seller_state")
    .agg(orders=("order_id", "count"), late=("delivery_delay_days", lambda x: (x > 0).sum()))
    .assign(pct_late=lambda d: (100 * d["late"] / d["orders"]).round(2))
    .sort_values("pct_late", ascending=False)
)
print("Late delivery rate by seller state (origin) — where shipments are sent from:")
print(late_by_origin.to_string())

Late delivery rate by customer state (destination) — where orders arrive:
                orders  late  pct_late
customer_state                        
AL                 396    85     21.46
MA                 713   125     17.53
SE                 332    51     15.36
PI                 475    66     13.89
CE                1273   176     13.83
RR                  40     5     12.50
BA                3253   396     12.17
RJ               12310  1495     12.14
PA                 942   106     11.25
ES                1992   214     10.74
PB                 516    54     10.47
TO                 274    27      9.85
MS                 701    68      9.70
PE                1587   153      9.64
RN                 470    44      9.36
SC                3537   291      8.23
GO                1950   128      6.56
RS                5327   325      6.10
MT                 885    53      5.99
DF                2074   118      5.69
MG               11319   519      4.59
SP               40399  1817 

**Conclusion — Geographic analysis**

- **Destination:** Some states have much higher late rates (e.g. AL, MA, SE, PI, CE). These may be harder to reach for carriers or have longer last-mile delivery times. Setting more accurate expectations or prioritising carrier coverage in these states could help.
- **Origin:** Sellers in certain states (e.g. AM, MA) also show higher late rates, which may reflect local logistics or dispatch delays. Focusing support on these regions could reduce late shipments closer to the source.
- **Combined view:** Looking at destination and origin together highlights which corridors (seller state → customer state) might need attention.

## 4. Seller Analysis

Orders are linked to sellers via order items, then each seller’s volume and late delivery rate are calculated. Sellers with fewer than 5 orders are excluded. “High-risk” sellers are those in the top quartile for both volume and late rate.

In [7]:
# Merge: items -> sellers, then join to orders_analysis (one row per order-item; we dedupe by order for order-level stats)
seller_orders = items[["order_id", "seller_id"]].merge(
    orders_analysis[["order_id", "delivery_delay_days"]], on="order_id", how="inner"
)
# One row per order-seller (an order can have multiple sellers)
seller_orders = seller_orders.drop_duplicates(subset=["order_id", "seller_id"])

# Seller-level metrics (sellers with at least 5 orders)
seller_stats = seller_orders.groupby("seller_id").agg(
    total_orders=("order_id", "nunique"),
    late_orders=("delivery_delay_days", lambda x: (x > 0).sum()),
).reset_index()
seller_stats["pct_late"] = (100 * seller_stats["late_orders"] / seller_stats["total_orders"]).round(2)

sellers_min_orders = seller_stats[seller_stats["total_orders"] >= 5].copy()
vol_p75 = sellers_min_orders["total_orders"].quantile(0.75)
late_p75 = sellers_min_orders["pct_late"].quantile(0.75)

problematic = sellers_min_orders[
    (sellers_min_orders["total_orders"] >= vol_p75) &
    (sellers_min_orders["pct_late"] >= late_p75)
].sort_values("pct_late", ascending=False)

print(f"Sellers with ≥5 orders: {len(sellers_min_orders)}")
print(f"High-volume threshold (p75): {vol_p75:.0f} orders")
print(f"High late-rate threshold (p75): {late_p75:.2f}%")
print(f"Problematic sellers (high volume and high late %): {len(problematic)}")
print("\nTop 10 by late rate (among problematic):")
print(problematic.head(10).to_string(index=False))

Sellers with ≥5 orders: 1759
High-volume threshold (p75): 46 orders
High late-rate threshold (p75): 10.00%
Problematic sellers (high volume and high late %): 87

Top 10 by late rate (among problematic):
                       seller_id  total_orders  late_orders  pct_late
54965bbe3e4f07ae045b90b0b8541f52            73           22     30.14
2a1348e9addc1af5aaa619b1a3679d6b            48           13     27.08
beadbee30901a7f61d031b6b686095ad            64           15     23.44
a49928bcdf77c55c6d6e05e09a9b4ca5            96           21     21.88
712e6ed8aa4aa1fa65dab41fed5737e4            77           16     20.78
06a2c3af7b3aee5d69171b0e14f0ee87           389           74     19.02
002100f778ceb8431b7a1020ff7ab48f            50            9     18.00
ea566164622c6b439516ab18062c42cd            50            9     18.00
cac4c8e7b1ca6252d8f20b2fc1a2e4af            73           13     17.81
6039e27294dc75811c0d8a39069f52c0            63           11     17.46


In [8]:
# Late delivery by seller state (using first seller per order for geography)
order_first_seller = items.groupby("order_id")["seller_id"].first().reset_index()
seller_geo = order_first_seller.merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
seller_geo = seller_geo.merge(
    orders_analysis[["order_id", "delivery_delay_days"]], on="order_id", how="inner"
)
state_late = seller_geo.groupby("seller_state").agg(
    orders=("order_id", "count"),
    late=("delivery_delay_days", lambda x: (x > 0).sum()),
).assign(pct_late=lambda d: (100 * d["late"] / d["orders"]).round(2))
state_late = state_late.sort_values("pct_late", ascending=False)
print("Late delivery rate by seller state (top 10):")
print(state_late.head(10))

Late delivery rate by seller state (top 10):
              orders  late  pct_late
seller_state                        
AM                 3     1     33.33
MA               388    74     19.07
RN                51     4      7.84
SP             68265  5014      7.34
RJ              4148   301      7.26
CE                85     6      7.06
MS                44     3      6.82
ES               307    19      6.19
PR              7395   400      5.41
DF               801    42      5.24


**Conclusion — Seller analysis**

- **Late delivery is concentrated in a subset of sellers.** Among sellers with at least 5 orders, a group stands out with both high volume and high late rates (top quartile in both). These are the natural first targets for support or process changes.
- **Geography plays a role too:** some states show clearly higher late rates at the seller level, pointing to regional logistics or carrier quality differences.
- **Most sellers perform well.** The problem isn’t universal, so targeted interventions (carrier selection, shipping deadlines, seller guidelines) are likely more effective than platform-wide changes.

## 5. Impact on Reviews

Orders are matched to reviews and classified by delay: **On-time** (≤0 days late), **Late** (1–7 days), **Very late** (8–30 days), **Extremely late** (>30 days). Then average rating and share of low ratings (1–2 stars) are compared across groups.

In [9]:
# Merge orders (with delay) and reviews
orders_reviews = orders_analysis[["order_id", "delivery_delay_days", "delivery_time_days"]].merge(
    reviews[["order_id", "review_score"]], on="order_id", how="inner"
)

# Delay category
def delay_category(days):
    if pd.isna(days):
        return "Missing"
    if days <= 0:
        return "On-time"
    if days <= 7:
        return "Late (1–7 days)"
    if days <= 30:
        return "Very late (8–30 days)"
    return "Extremely late (>30 days)"

orders_reviews["delay_category"] = orders_reviews["delivery_delay_days"].apply(delay_category)

# Metrics by category
review_by_delay = orders_reviews.groupby("delay_category").agg(
    n_reviews=("review_score", "count"),
    avg_score=("review_score", "mean"),
    low_rating_count=("review_score", lambda x: (x <= 2).sum()),
).assign(
    pct_low_rating=lambda d: (100 * d["low_rating_count"] / d["n_reviews"]).round(2)
)
# Order for display
order_cat = ["On-time", "Late (1–7 days)", "Very late (8–30 days)", "Extremely late (>30 days)", "Missing"]
review_by_delay = review_by_delay.reindex([c for c in order_cat if c in review_by_delay.index])
print(review_by_delay.to_string())

                           n_reviews  avg_score  low_rating_count  pct_low_rating
delay_category                                                                   
On-time                        89681   4.290730              8304            9.26
Late (1–7 days)                 3611   2.713099              1785           49.43
Very late (8–30 days)           2465   1.652333              1989           80.69
Extremely late (>30 days)        330   2.057576               224           67.88


In [10]:
# Compare problematic vs non-problematic sellers (using same definition as in Seller Analysis)
order_seller_one = items.groupby("order_id")["seller_id"].first().reset_index()
orders_reviews_with_seller = orders_reviews.merge(order_seller_one, on="order_id", how="left")
problematic_ids = set(problematic["seller_id"])
orders_reviews_with_seller["problematic_seller"] = orders_reviews_with_seller["seller_id"].isin(problematic_ids)

seller_review_compare = orders_reviews_with_seller.groupby("problematic_seller").agg(
    n_reviews=("review_score", "count"),
    avg_score=("review_score", "mean"),
    low_pct=("review_score", lambda x: (100 * (x <= 2).sum() / len(x)).round(2)),
).rename(index={False: "Other sellers", True: "Problematic sellers"})
print(seller_review_compare)

                     n_reviews  avg_score  low_pct
problematic_seller                                
Other sellers            85225   4.184394    12.08
Problematic sellers      10862   3.933990    18.47


**Conclusion — Impact on reviews**

- **Late delivery is strongly associated with worse reviews.** On-time orders average around 4.3 stars with ~9% low ratings. As delay increases, scores drop and low-rating share rises sharply: for “Very late” orders, the average is around 1.65 and over 80% of reviews are 1–2 stars.
- **The relationship is monotonic:** the worse the delay, the worse the ratings. This supports treating delivery performance as a direct lever for improving customer satisfaction.
- **High-risk sellers (high volume + high late rate) also receive worse reviews on average** — late delivery doesn’t just affect the platform’s image, it directly hurts the sellers who are most often responsible.
- **Takeaway:** Reducing late deliveries should be treated as a direct lever for improving review scores and, over time, retention and trust.

## 6. Summary and Recommendations

**What came out of the analysis:**

1. **Logistics:** About 93% of delivered orders arrive on or before the estimated date. The remaining ~7% are late, with a small fraction extremely so (>30 days). Performance varies by month but doesn’t show a single consistent trend.

2. **Geography:** Late delivery rates vary significantly by region — both by customer state (destination) and seller state (origin). Some states show much higher late rates, pointing to corridors worth prioritising for carrier or fulfilment improvements.

3. **Sellers:** A subset combines high order volume with high late rates. These sellers are the obvious focus for support, carrier selection, or fulfilment rules. Late rates also differ by seller state, suggesting regional logistics quality plays a role.

4. **Reviews:** There’s a clear link between late delivery and worse reviews. On-time orders get much higher scores and far fewer 1–2 star ratings. Sellers with the worst delivery performance receive worse ratings on average.

**Recommended next steps:**

- **Reduce late deliveries** by targeting the 87 high-volume, high-late-rate sellers and the states with the highest late rates (carrier options, cut-off times, or seller guidelines).
- **Review estimated delivery dates:** many orders arrive early, so estimates may already be conservative. Keep monitoring to make sure they stay accurate and don’t drift.
- **Use reviews as a signal:** track low-rating trends for the high-risk seller group after any interventions, and flag suspicious patterns early.

## 7. Interventions and what to measure

Below are more concrete actions and metrics. The goal is to turn the EDA findings into something operational.

### On the 87 high-risk sellers

**Who:** Sellers in the top quartile for both order volume and late delivery rate (identified in Section 4).

**Possible actions:** require use of preferred carriers; tighten shipping cut-off times; dedicated support for packing and dispatch; review visibility or promotion until late rate falls below a target threshold.

**Goal:** Bring this group’s late rate toward the platform median (e.g. from ~10%+ down to <7%) within 2–3 months.

### On critical states

**Who:** Destination states with the highest late rates (AL, MA, SE, PI, CE from Section 3) and origin states where sellers show elevated late rates (e.g. AM, MA).

**Possible actions:** partner with or prioritise carriers with better coverage in these states; adjust estimated delivery dates by corridor where the data support it; run a pilot in 1–2 critical states before scaling.

**Goal:** Bring the worst-performing states closer to the national average.

### What to track after intervening

Track the same metrics used here, before vs after:
- % of delivered orders that are late (overall, by seller, by state)
- Share of 1–2 star reviews (overall and for the previously high-risk sellers)
- Average review score by delay category and seller segment

The key KPIs to watch: **late delivery rate** (target: reduce from ~6.8% to <5% in 6 months), **low-rating share** (target: reduce by ~1–2 pp overall), and **late rate for the 87 high-risk sellers** (target: <7% within 3 months of intervention). Review these at least monthly; consider weekly tracking for the high-risk seller group in the first quarter post-intervention.